In [12]:
%pip install einops

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
import os
import torch

# Add the correct nested paths to sys.path
sys.path.append(os.path.join(os.getcwd(), 'vggt'))
sys.path.append(os.path.join(os.getcwd(), 'vggt', 'vggt'))

try:
    # This matches the 'vggt.py' file you found in the models folder
    from models.vggt import VGGT
    print("SUCCESS: VGGT class loaded from models.vggt")
except ImportError:
    from vggt.models.vggt import VGGT
    print("SUCCESS: VGGT class loaded from vggt.models.vggt")

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VGGT().to(device)

# Load your pretrained model.pt
model.load_state_dict(torch.load('model_folder/model.pt', map_location=device))
model.eval()
print("Model weights loaded successfully.")

c:\Users\rajve\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FlashAttention not found. Falling back to PyTorch scaled_dot_product_attention.
xFormers not found. Using standard attention fallback.
SUCCESS: VGGT class loaded from models.vggt


C:\Users\rajve\AppData\Local\Temp\ipykernel_14712\122462479.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model_folder/model.pt', ma

Model weights loaded successfully.


In [4]:
import sys
import os

# This adds the 'vggt' folder to Python's search list
sys.path.append(os.path.join(os.getcwd(), 'vggt'))

# Now try the import again
try:
    from vggt.utils.geometry import get_derived_point_map_torch
    print("SUCCESS: Geometry module loaded!")
except ImportError:
    # If the above fails, try the direct path
    from utils.geometry import get_derived_point_map_torch
    print("SUCCESS: Geometry module loaded from utils!")

SUCCESS: Geometry module loaded!


In [20]:
import os
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
import glob

# 1. AUTO-FIND THE DEPTH FILE
# This looks into your co3d_data folder for ANY depth png
search_path = 'co3d_data/cup/*/depths/*.png'
found_files = glob.glob(search_path)

if not found_files:
    # If that fails, let's try searching the current directory
    found_files = glob.glob('**/depths/*.png', recursive=True)

if found_files:
    gt_depth_path = found_files[0]
    print(f"✅ Found Ground Truth: {gt_depth_path}")
    
    # 2. LOAD AND NORMALIZE
    gt_depth_pil = Image.open(gt_depth_path).convert('L') # Convert to Grayscale
    gt_depth_tensor = T.ToTensor()(gt_depth_pil).unsqueeze(0)
    gt_depth_tensor = F.interpolate(gt_depth_tensor, size=(224, 224))
    
    # Ensure it's on the same device as your model output
    gt_depth_tensor = gt_depth_tensor.to(d_pred.device)

    # 3. CALCULATE THE TWO DIFFERENT LOSSES
    # Supervised = Model vs. The File we just found
    loss_supervised = F.mse_loss(d_pred, gt_depth_tensor).item()

    # Consistency = Model vs. Its own Geometry (p_math)
    loss_consistency = F.mse_loss(p_pred, p_math).item()

    # 4. FINAL REPORT
    print(f"\n" + "="*40)
    print(f"       PHASE 2: THE REAL DIFFERENCE")
    print(f"="*40)
    print(f"SUPERVISED LOSS (Accuracy):   {loss_supervised:.6f}")
    print(f"CONSISTENCY LOSS (Logic):    {loss_consistency:.6f}")
    print(f"="*40)
    print(f"The 'Flaw' is the Consistency Loss. It proves the model's")
    print(f"internal 3D logic is broken even if it looks like a cup.")
    print(f"="*40)
else:
    print("❌ Still couldn't find a depth file. Please check your folder name!")

✅ Found Ground Truth: co3d_data/cup\620_101304_203141\depths\frame000001.jpg.geometric.png

       PHASE 2: THE REAL DIFFERENCE
SUPERVISED LOSS (Accuracy):   0.196512
CONSISTENCY LOSS (Logic):    0.063882
The 'Flaw' is the Consistency Loss. It proves the model's
internal 3D logic is broken even if it looks like a cup.


In [13]:
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import glob
import os

# --- 1. GEOMETRY HELPER (The 'Math' Judge) ---
def calculate_p_math(depth, intrinsics):
    B, _, H, W = depth.shape
    # Create coordinate grid
    y, x = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij')
    pixels = torch.stack([x, y, torch.ones_like(x)], dim=0).reshape(1, 3, -1).to(depth.device)
    
    # Default intrinsics if None (Standard pinhole model)
    if intrinsics is None:
        intrinsics = torch.tensor([[0.7, 0, 0.5], [0, 0.7, 0.5], [0, 0, 1]]).unsqueeze(0).to(depth.device)
    
    inv_K = torch.inverse(intrinsics.float())
    rays = torch.bmm(inv_K, pixels) 
    return (rays * depth.reshape(B, 1, -1)).reshape(B, 3, H, W)

# --- 2. DATASET DEFINITION ---
class CO3DRefinementDataset(Dataset):
    def __init__(self, root_dir):
        self.img_files = sorted(glob.glob(os.path.join(root_dir, "**/images/*.jpg"), recursive=True))
        self.depth_files = sorted(glob.glob(os.path.join(root_dir, "**/depths/*.png"), recursive=True))
        self.transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    def __len__(self): 
        return min(len(self.img_files), len(self.depth_files))

    def __getitem__(self, idx):
        img = self.transform(Image.open(self.img_files[idx]).convert('RGB'))
        depth = self.transform(Image.open(self.depth_files[idx]).convert('L'))
        return {'image': img, 'depth_gt': depth}

# --- 3. EXECUTION ---
dataset = CO3DRefinementDataset('co3d_data/cup')
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

total_loss_sup, total_loss_geo, count = 0, 0, 0
MAX_SAMPLES = 100 



with torch.no_grad():
    pbar = tqdm(dataloader, total=min(len(dataloader), MAX_SAMPLES))
    for batch in pbar:
        if count >= MAX_SAMPLES: break
        
        img = batch['image'].unsqueeze(1).to(device) 
        gt_depth = batch['depth_gt'].to(device)      
        
        preds = model(img)
        
        p_pred = preds['world_points'].squeeze(1).permute(0, 3, 1, 2).float()
        d_pred = preds['depth'].squeeze(1).permute(0, 3, 1, 2).float()
        
        # Calculate consistency
        loss_sup = torch.nn.functional.mse_loss(d_pred, gt_depth).item()
        p_math = calculate_p_math(d_pred, preds.get('intrinsics'))
        loss_geo = torch.nn.functional.mse_loss(p_pred, p_math).item()
        
        total_loss_sup += loss_sup
        total_loss_geo += loss_geo
        count += 1
        pbar.set_postfix({"MSE": f"{loss_sup:.4f}", "Geo_Err": f"{loss_geo:.4f}"})

# --- 4. OUTPUT TABLE ---
if count > 0:
    avg_sup, avg_geo = total_loss_sup / count, total_loss_geo / count
    df_matrix = pd.DataFrame([{
        "Total_Samples": count,
        "Depth_MSE": round(avg_sup, 6),
        "Geo_Inconsistency": round(avg_geo, 6),
        "Geometric_Accuracy_%": round(max(0, 100 - (avg_geo * 100)), 2)
    }])
    print("\n" + "="*50 + "\n FINAL EVALUATION MATRIX \n" + "="*50)
    print(df_matrix.to_string(index=False))


100%|██████████| 100/100 [07:13<00:00,  4.34s/it, MSE=0.8042, Geo_Err=0.0971]



 FINAL EVALUATION MATRIX 
 Total_Samples  Depth_MSE  Geo_Inconsistency  Geometric_Accuracy_%
           100    0.27261           0.074763                 92.52


In [5]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import glob

class CO3DCupDataset(Dataset):
    def __init__(self, root_dir, img_size=(224, 224)):
        self.img_size = img_size
        self.samples  = []

        scene_dirs = sorted([
            d for d in glob.glob(os.path.join(root_dir, "*"))
            if os.path.isdir(d)
        ])
        print(f"Found {len(scene_dirs)} scene folders")

        for scene_dir in scene_dirs:
            img_dir        = os.path.join(scene_dir, "images")
            depth_dir      = os.path.join(scene_dir, "depths")
            depth_mask_dir = os.path.join(scene_dir, "depth_masks")
            mask_dir       = os.path.join(scene_dir, "masks")

            if not os.path.exists(img_dir):
                continue

            img_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

            for img_path in img_files:
                basename   = os.path.splitext(os.path.basename(img_path))[0]
                depth_path = os.path.join(depth_dir, basename + ".jpg.geometric.png")

                if not os.path.exists(depth_path):
                    continue

                mask_path       = os.path.join(mask_dir,       basename + ".png")
                depth_mask_path = os.path.join(depth_mask_dir, basename + ".png")

                self.samples.append({
                    "image":      img_path,
                    "depth":      depth_path,
                    "mask":       mask_path       if os.path.exists(mask_path)       else None,
                    "depth_mask": depth_mask_path if os.path.exists(depth_mask_path) else None,
                    "scene":      os.path.basename(scene_dir),
                })

        print(f"✅ Total valid samples: {len(self.samples)}")

    def load_depth(self, path):
        # ── CONFIRMED FIX: uint16 / 10000 = metres ────────
        raw   = np.array(Image.open(path), dtype=np.float32)
        depth = raw / 10000.0

        # Sanity clip for tabletop cup (0.1m – 5m)
        depth[depth < 0.1] = 0.0
        depth[depth > 5.0] = 0.0
        return depth

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # ── Image ─────────────────────────────────────────
        img = Image.open(s["image"]).convert("RGB")
        img = img.resize(self.img_size, Image.BILINEAR)
        img = torch.tensor(np.array(img), dtype=torch.float32)
        img = img.permute(2, 0, 1) / 255.0

        # ── Depth ─────────────────────────────────────────
        depth = self.load_depth(s["depth"])
        depth = torch.tensor(depth, dtype=torch.float32).unsqueeze(0)
        depth = torch.nn.functional.interpolate(
            depth.unsqueeze(0), size=self.img_size, mode='nearest'
        ).squeeze(0)

        out = {"image": img, "depth_gt": depth, "scene": s["scene"]}

        # ── Masks ─────────────────────────────────────────
        if s["mask"]:
            m = Image.open(s["mask"]).convert("L").resize(self.img_size)
            out["mask"] = torch.tensor(
                np.array(m), dtype=torch.float32
            ).unsqueeze(0) / 255.0

        if s["depth_mask"]:
            dm = Image.open(s["depth_mask"]).convert("L").resize(self.img_size)
            out["depth_mask"] = torch.tensor(
                np.array(dm), dtype=torch.float32
            ).unsqueeze(0) / 255.0

        return out


# ── VERIFY scale is fixed ─────────────────────────────────
# Quick raw check BEFORE loading dataset
test_file = sorted(glob.glob("co3d_data/cup/*/depths/*.jpg.geometric.png"))[0]
raw = np.array(Image.open(test_file), dtype=np.float32)
print(f"RAW max pixel:    {raw.max()}")
print(f"After /10000:     {raw[raw>0].mean()/10000:.4f} m  ← should be ~1–3m")

# ── Load dataset ──────────────────────────────────────────
dataset    = CO3DCupDataset("co3d_data/cup/", img_size=(224, 224))
dataloader = DataLoader(dataset, batch_size=1, shuffle=True,
                        num_workers=0, pin_memory=True)

# ── Confirm ───────────────────────────────────────────────
print("\nDepth verification:")
for i in range(4):
    batch = next(iter(dataloader))
    depth = batch['depth_gt'][0, 0].numpy()
    valid = depth[depth > 0]
    if len(valid) > 0:
        print(f"  Sample {i+1}: min={valid.min():.3f}m  "
              f"max={valid.max():.3f}m  mean={valid.mean():.3f}m")
    else:
        print(f"  Sample {i+1}: ⚠️ all zeros after clipping!")

RAW max pixel:    24205.0
After /10000:     1.9347 m  ← should be ~1–3m
Found 24 scene folders
✅ Total valid samples: 4132

Depth verification:
  Sample 1: min=1.669m  max=2.216m  mean=1.872m
  Sample 2: min=1.982m  max=2.084m  mean=2.025m
  Sample 3: min=1.957m  max=2.250m  mean=2.023m
  Sample 4: min=1.905m  max=2.262m  mean=2.021m


In [14]:
import torch
import pandas as pd
from tqdm import tqdm
import time

# --- SETUP ---
NUM_EPOCHS = 10
MAX_SAMPLES_PER_EPOCH = 200 
epoch_results = []

print(f"🔄 Starting 10-Epoch Evaluation on {device}...")

for epoch in range(1, NUM_EPOCHS + 1):
    total_loss_sup, total_loss_geo, count = 0, 0, 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", total=MAX_SAMPLES_PER_EPOCH)
    
    with torch.no_grad():
        for batch in pbar:
            if count >= MAX_SAMPLES_PER_EPOCH: break
            
            img = batch['image'].unsqueeze(1).to(device) 
            gt_depth = batch['depth_gt'].to(device)      
            
            preds = model(img)
            
            p_pred = preds['world_points'].squeeze(1).permute(0, 3, 1, 2).float()
            d_pred = preds['depth'].squeeze(1).permute(0, 3, 1, 2).float()
            
            # Metrics
            loss_sup = torch.nn.functional.mse_loss(d_pred, gt_depth).item()
            p_math = calculate_p_math(d_pred, preds.get('intrinsics'))
            loss_geo = torch.nn.functional.mse_loss(p_pred, p_math).item()
            
            total_loss_sup += loss_sup
            total_loss_geo += loss_geo
            count += 1
            pbar.set_postfix({"MSE": f"{loss_sup:.4f}", "Geo": f"{loss_geo:.4f}"})

    # Record Epoch Stats
    avg_sup = total_loss_sup / count
    avg_geo = total_loss_geo / count
    accuracy = max(0, 100 - (avg_geo * 100))
    
    epoch_results.append({
        "Epoch": epoch,
        "Depth_MSE": round(avg_sup, 6),
        "Geo_Error": round(avg_geo, 6),
        "Accuracy_%": round(accuracy, 2)
    })

# --- FINAL TABLE ---
df_final = pd.DataFrame(epoch_results)
print("\n" + "="*60)
print("📊 10-EPOCH CONSOLIDATED EVALUATION")
print("="*60)
print(df_final.to_string(index=False))

# Calculate Grand Average
grand_avg_acc = df_final["Accuracy_%"].mean()
print(f"\n✅ Final Consolidated Geometric Accuracy: {grand_avg_acc:.2f}%")

🔄 Starting 10-Epoch Evaluation on cuda...


Epoch 1/10:   0%|          | 0/200 [00:00<?, ?it/s]e:\vggt_project\vggt\vggt\models\vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
Epoch 10/10: 100%|██████████| 200/200 [14:15<00:00,  4.28s/it, MSE=0.2362, Geo=0.1061]



📊 10-EPOCH CONSOLIDATED EVALUATION
 Epoch  Depth_MSE  Geo_Error  Accuracy_%
     1   0.240026   0.073838       92.62
     2   0.312979   0.073875       92.61
     3   0.287481   0.074018       92.60
     4   0.269608   0.074458       92.55
     5   0.268485   0.075668       92.43
     6   0.271860   0.076528       92.35
     7   0.314801   0.075283       92.47
     8   0.270570   0.074720       92.53
     9   0.251978   0.075671       92.43
    10   0.302804   0.076222       92.38

✅ Final Consolidated Geometric Accuracy: 92.50%


In [10]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def process_points(points, conf, colors_hwc):
    H, W = points.shape[:2]
    x, y, z = points[...,0].flatten(), points[...,1].flatten(), points[...,2].flatten()
    rgb = colors_hwc.reshape(-1, 3)
    c   = conf.flatten()

    # ── Step 1: confidence filter (keep top 60%) ───────────────────────────
    mask      = c > np.percentile(c, 40)
    x,y,z,rgb = x[mask], y[mask], z[mask], rgb[mask]

    # ── Step 2: remove depth outliers only (NO slab/variance filter) ───────
    z_lo, z_hi = np.percentile(z, 1), np.percentile(z, 99)
    mask2      = (z > z_lo) & (z < z_hi)
    x,y,z,rgb  = x[mask2], y[mask2], z[mask2], rgb[mask2]

    # ── Step 3: subsample ──────────────────────────────────────────────────
    step      = max(1, len(x) // 30000)   # cap at 30k points for speed
    x,y,z,rgb = x[::step], y[::step], z[::step], rgb[::step]

    # ── Step 4: FIX colors — normalize properly ────────────────────────────
    rgb = rgb.astype(np.float32)
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    rgb = np.clip(rgb, 0.0, 1.0)

    # Build color list
    r = (rgb[:, 0] * 255).astype(np.uint8)
    g = (rgb[:, 1] * 255).astype(np.uint8)
    b = (rgb[:, 2] * 255).astype(np.uint8)
    colors = [f'rgb({ri},{gi},{bi})' for ri, gi, bi in zip(r, g, b)]

    return x, y, z, colors


def visualize_image_and_3d(
    dataset,
    device,
    image_idx = 0,
    save_path = "cup_3d.html"
):
    model.eval()

    batch = dataset[image_idx]

    # ── Get image tensor for model ─────────────────────────────────────────
    img_tensor = batch['image'].unsqueeze(0).unsqueeze(0).to(device).float()

    # ── Get display image ──────────────────────────────────────────────────
    img_raw = batch['image'].cpu().numpy()          # could be (3,H,W) or (H,W,3)
    if img_raw.ndim == 3 and img_raw.shape[0] == 3:
        img_raw = img_raw.transpose(1, 2, 0)        # → (H, W, 3)
    img_raw = img_raw.astype(np.float32)
    if img_raw.max() > 1.0:
        img_raw = img_raw / 255.0

    print(f"Image shape : {img_raw.shape}  range: {img_raw.min():.2f}–{img_raw.max():.2f}")

    # ── Inference ──────────────────────────────────────────────────────────
    with torch.no_grad():
        preds  = model(img_tensor)
        points = preds['world_points'].squeeze().cpu().numpy()       # (H, W, 3)
        conf   = preds['world_points_conf'].squeeze().cpu().numpy()  # (H, W)

    print(f"Points shape: {points.shape}")

    # ── Resize image to match point cloud spatial dims if needed ───────────
    H_p, W_p = points.shape[:2]
    H_i, W_i = img_raw.shape[:2]
    if (H_p, W_p) != (H_i, W_i):
        import cv2
        img_raw = cv2.resize(img_raw, (W_p, H_p), interpolation=cv2.INTER_LINEAR)
        print(f"Resized image → {img_raw.shape}")

    x, y, z, colors = process_points(points, conf, img_raw.copy())
    print(f"Plotting {len(x):,} points")

    # ── Verify a sample of colors ──────────────────────────────────────────
    print(f"Sample colors: {colors[:3]}")

    # ── Layout ─────────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.35, 0.65],
        subplot_titles=("📷  Input Image", "🔵  3D Reconstruction"),
        specs=[[{"type": "image"}, {"type": "scene"}]],
        horizontal_spacing=0.02,
    )

    # Real image
    fig.add_trace(
        go.Image(z=(img_raw * 255).astype(np.uint8)),
        row=1, col=1
    )

    # 3D point cloud — pass colors as a list directly
    fig.add_trace(
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            name='Point Cloud',
            marker=dict(
                size=2,
                color=colors,       # list of 'rgb(r,g,b)' strings
                opacity=0.95,
            )
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text=f"VGGT — 3D Reconstruction | Scene: {batch['scene']}",
            font=dict(size=18, color='white')
        ),
        paper_bgcolor='rgb(15,25,50)',
        font=dict(color='white'),
        scene=dict(
            bgcolor='rgb(10,20,40)',
            xaxis=dict(title='X (m)', gridcolor='rgb(50,70,100)', color='white'),
            yaxis=dict(title='Y (m)', gridcolor='rgb(50,70,100)', color='white'),
            zaxis=dict(title='Z (m)', gridcolor='rgb(50,70,100)', color='white'),
            aspectmode='data',     # keeps true proportions — no stretching
            camera=dict(
                eye=dict(x=1.2, y=-1.2, z=0.8),
                up=dict(x=0, y=0, z=1)
            )
        ),
        height=680, width=1300,
        margin=dict(l=0, r=0, b=0, t=60),
    )
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)

    fig.write_html(save_path)
    print(f"✅ Saved → {save_path}")
    return fig


# ── Run ────────────────────────────────────────────────────────────────────────
fig = visualize_image_and_3d(
    dataset   = dataset,
    device    = device,
    image_idx = 0,
    save_path = "cup_3d.html"
)

Image shape : (224, 224, 3)  range: 0.00–1.00


e:\vggt_project\vggt\vggt\models\vggt.py:65: FutureWarning:

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.



Points shape: (224, 224, 3)
Plotting 29,501 points
Sample colors: ['rgb(107,140,150)', 'rgb(106,137,143)', 'rgb(113,148,157)']
✅ Saved → cup_3d.html


In [9]:
%pip install flash-attn --no-build-isolation

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
     - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
     -- ------------------------------------- 0.5/8.4 MB 1.9 MB/s eta 0:00:05
     --- ------------------------------------ 0.8/8.4 MB 1.6 MB/s eta 0:00:05
     ---- ----------------------------------- 1.0/8.4 MB 1.5 MB/s eta 0:00:05
     ------ --------------------------------- 1.3/8.4 MB 1.4 MB/s eta 0:00:06
     ------- -------------------------------- 1.6/8.4 MB 1.1 MB/s eta 0:00:06
     ------- -------------------------------- 1.6/8.4 MB 1.1 MB/s eta 0:00:06
     -------- ------------------------------- 1.8/8.4 MB 1.0 MB/s eta 0:00:07
     -------- ------------------------------- 1.8/8.4 MB 1.0 MB/s eta 0:00:07
     --------- ------------------------------ 2.1/8.4 MB 1.0 MB/s eta 0:00:07
     --------- ------------------------------ 2.1/8.4 MB 1.0 MB/s eta 0:00:07
     

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\rajve\\AppData\\Local\\Temp\\pip-install-ay042gv0\\flash-attn_56b9bbae06f7451c92b5cfb0a6a17a94\\csrc/composable_kernel/library/include/ck/library/tensor_operation_instance/gpu/grouped_conv_bwd_weight/device_grouped_conv_bwd_weight_two_stage_xdl_instance.hpp'


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import torch
print(torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

NVIDIA GeForce RTX 3050 Laptop GPU
Compute capability: (8, 6)


In [5]:
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from torch.amp import autocast, GradScaler

# ─────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE   = 4
DIM          = 768
NUM_HEADS    = 12
STEPS        = 30
WARMUP       = 10
SEQ_LEN_LIST = [1024, 2048, 4096, 8192]

# ─────────────────────────────────────────────────────────
# ACCURACY (RELATIVE ERROR BASED)
# ─────────────────────────────────────────────────────────
def compute_accuracy(pred, target):
    rel_error = torch.abs(pred - target) / (torch.abs(target) + 1e-6)
    return (rel_error < 0.1).float().mean().item() * 100

# ─────────────────────────────────────────────────────────
# FLOPs (More realistic attention scaling)
# ─────────────────────────────────────────────────────────
def calculate_flops(b, l, d):
    return 6 * b * (l**2) * d

# ─────────────────────────────────────────────────────────
# LOAD VGGT ATTENTION (fallback safe)
# ─────────────────────────────────────────────────────────
try:
    from vggt.layers.attention import Attention, MemEffAttention
except ImportError:
    print(" VGGT not found, using dummy attention")

    class Attention(nn.Module):
        def __init__(self, dim, num_heads):
            super().__init__()
            self.net = nn.Linear(dim, dim)

        def forward(self, x):
            return self.net(x)

    MemEffAttention = Attention

# ─────────────────────────────────────────────────────────
# BENCHMARK FUNCTION
# ─────────────────────────────────────────────────────────
def benchmark(model_class, seq_len):
    try:
        model     = model_class(dim=DIM, num_heads=NUM_HEADS).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        scaler    = GradScaler()
        criterion = nn.MSELoss()

        x = torch.randn(BATCH_SIZE, seq_len, DIM, device=DEVICE)
        y = torch.randn(BATCH_SIZE, seq_len, DIM, device=DEVICE)

        # ── Warmup ─────────────────────────────────────────
        for _ in range(WARMUP):
            with autocast(device_type='cuda', dtype=torch.float16):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)

        losses, accs = [], []

        # ── Timed Run ──────────────────────────────────────
        start.record()
        for _ in range(STEPS):
            optimizer.zero_grad()

            with autocast(device_type='cuda', dtype=torch.float16):
                out  = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            losses.append(loss.item())
            accs.append(compute_accuracy(out.detach(), y))

        end.record()
        torch.cuda.synchronize()

        elapsed = start.elapsed_time(end) / 1000  # seconds

        return {
            "latency_ms": (elapsed / STEPS) * 1000,
            "memory_mb": torch.cuda.max_memory_allocated() / 1024**2,
            "tflops": (calculate_flops(BATCH_SIZE, seq_len, DIM) * STEPS / elapsed) / 1e12,

            # ML metrics
            "loss_mean": np.mean(losses),
            "loss_std":  np.std(losses),
            "acc_mean":  np.mean(accs),
            "acc_std":   np.std(accs)
        }

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return None
        raise e

# ─────────────────────────────────────────────────────────
# RUN BENCHMARK
# ─────────────────────────────────────────────────────────
print(f"\n Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}\n")

rows = []

for seq_len in SEQ_LEN_LIST:
    print(f" seq_len={seq_len} ...", end=" ")

    van = benchmark(Attention, seq_len)
    opt = benchmark(MemEffAttention, seq_len)

    if van and opt:
        row = {
            "SeqLen": seq_len,

            # System comparison
            "Latency_Diff_ms": van["latency_ms"] - opt["latency_ms"],
            "Memory_Diff_MB":  van["memory_mb"] - opt["memory_mb"],
            "TFLOPS_Diff":     opt["tflops"] - van["tflops"],

            # Raw values (important for paper)
            "Latency_Van": van["latency_ms"],
            "Latency_Opt": opt["latency_ms"],

            # ML metrics
            "Loss_Van": van["loss_mean"],
            "Loss_Opt": opt["loss_mean"],
            "Loss_std": van["loss_std"],

            "Acc_Van": van["acc_mean"],
            "Acc_Opt": opt["acc_mean"],
            "Acc_std": van["acc_std"],
        }

        rows.append(row)
        print(" Done")

    else:
        print(" OOM")

    torch.cuda.empty_cache()

df = pd.DataFrame(rows)

# ─────────────────────────────────────────────────────────
# PRINT RESULTS
# ─────────────────────────────────────────────────────────
print("\n FINAL RESULTS")
print(df.round(4).to_string(index=False))

# ─────────────────────────────────────────────────────────
# SAVE RESULTS (VERY IMPORTANT FOR PROJECT)
# ─────────────────────────────────────────────────────────
df.to_csv("benchmark_results.csv", index=False)
print("\n Results saved to benchmark_results.csv")

# ─────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────
plt.figure()
plt.plot(df["SeqLen"], df["Latency_Diff_ms"], marker='o', label="Latency Diff")
plt.plot(df["SeqLen"], df["TFLOPS_Diff"], marker='o', label="TFLOPS Diff")
plt.axhline(0, linestyle='--')
plt.title("System Performance vs Sequence Length")
plt.xlabel("Sequence Length")
plt.ylabel("Difference")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(df["SeqLen"], df["Loss_Van"], marker='o', label="Vanilla")
plt.plot(df["SeqLen"], df["Loss_Opt"], marker='o', linestyle='--', label="Optimized")

# Stability shading
plt.fill_between(df["SeqLen"],
                 df["Loss_Van"] - df["Loss_std"],
                 df["Loss_Van"] + df["Loss_std"],
                 alpha=0.2)

plt.title("Loss Comparison with Stability")
plt.xlabel("Sequence Length")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


 Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

 seq_len=1024 ... 

KeyboardInterrupt: 